In [ ]:
import torch, torchvision
import torch.nn as nn, torch.optim as optim
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset

t   = T.ToTensor()
trd = DataLoader(Subset(torchvision.datasets.CIFAR10('./data', True,  transform=t, download=True), range(200)), 10, True)
ted = DataLoader(Subset(torchvision.datasets.CIFAR10('./data', False, transform=t, download=True), range(50)),  10)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(4096,512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512,10)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
def train(model, opt, crit, epochs):
    for e in range(epochs):
        model.train()
        rl = 0
        tc = 0
        tn = 0
        for x, y in trd:
            opt.zero_grad()
            o = model(x)
            l = crit(o, y)
            l.backward()
            opt.step()
            rl += l.item()
            tc += (o.argmax(1)==y).sum().item()
            tn += len(y)
        model.eval()
        tl = 0
        c  = 0
        n  = 0
        with torch.no_grad():
            for x, y in ted:
                o = model(x)
                l = crit(o, y)
                tl += l.item()
                c  += (o.argmax(1)==y).sum().item()
                n  += len(y)
        print(f'E{e+1} | Train Loss: {rl/len(trd):.2f} Acc: {tc/tn*100:.1f}% | Test Loss: {tl/len(ted):.2f} Acc: {c/n*100:.1f}%')

model = CNN()
opt   = optim.Adam(model.parameters(), 1e-3)
crit  = nn.CrossEntropyLoss()
train(model, opt, crit, 30)